<a href="https://colab.research.google.com/github/natifloresgz-ui/TelecomX_LATAM/blob/main/01_carga_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**📌 Extracción**

In [28]:

import requests
import pandas as pd

# Cargar datos desde la API
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"
response = requests.get(url)
data = response.json()
df = pd.json_normalize(data)

df.head()

,customerID,Churn,customer.gender,customer.SeniorCitizen,customer.Partner,customer.Dependents,customer.tenure,phone.PhoneService,phone.MultipleLines,internet.InternetService,...,internet.OnlineBackup,internet.DeviceProtection,internet.TechSupport,internet.StreamingTV,internet.StreamingMovies,account.Contract,account.PaperlessBilling,account.PaymentMethod,account.Charges.Monthly,account.Charges.Total
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.3
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.4
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.4


🔧 **Transformación**


In [7]:
df.isnull().sum()

,0
customerID,0
Churn,0
customer.gender,0
customer.SeniorCitizen,0
customer.Partner,0
customer.Dependents,0
customer.tenure,0
phone.PhoneService,0
phone.MultipleLines,0
internet.InternetService,0


In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df = df.drop_duplicates()

In [11]:
df["Churn"].unique()

array(['No', 'Yes', ''], dtype=object)

In [12]:
df["account.Contract"].unique()

array(['One year', 'Month-to-month', 'Two year'], dtype=object)

In [17]:
df["internet.InternetService"].unique()
df["account.PaymentMethod"].unique()
df["phone.PhoneService"].unique()

array(['Yes', 'No'], dtype=object)

In [18]:
df.dtypes

,0
customerID,object
Churn,object
customer.gender,object
customer.SeniorCitizen,int64
customer.Partner,object
customer.Dependents,object
customer.tenure,int64
phone.PhoneService,object
phone.MultipleLines,object
internet.InternetService,object


In [19]:
df["account.Charges.Total"] = pd.to_numeric(
    df["account.Charges.Total"],
    errors="coerce"
)

In [20]:
df.isnull().sum()

,0
customerID,0
Churn,0
customer.gender,0
customer.SeniorCitizen,0
customer.Partner,0
customer.Dependents,0
customer.tenure,0
phone.PhoneService,0
phone.MultipleLines,0
internet.InternetService,0


In [21]:
df["account.Charges.Total"] = df["account.Charges.Total"].fillna(0)

### Limpieza y validación de datos

Se revisaron valores ausentes, registros duplicados, tipos de datos y consistencia
en variables categóricas.  
Se corrigieron errores de formato en variables numéricas y se trataron valores
nulos para asegurar la calidad de los datos antes del análisis.

In [24]:
cols_cat = df.select_dtypes(include="object").columns

df[cols_cat] = df[cols_cat].apply(lambda x: x.str.strip().str.lower())

In [23]:
df["Churn"].unique()

array(['no', 'yes', ''], dtype=object)

In [25]:
df["Churn"] = df["Churn"].replace({"yes": 1, "no": 0})

In [26]:
df.replace(
    {
        "no internet service": "no",
        "no phone service": "no"
    },
    inplace=True
)

In [27]:
df["account.auto_payment"] = df["account.PaymentMethod"].str.contains("automatic").astype(int)

In [29]:
df["account.Contract"].unique()
df["internet.InternetService"].unique()
df["account.PaymentMethod"].unique()

array(['Mailed check', 'Electronic check', 'Credit card (automatic)',
       'Bank transfer (automatic)'], dtype=object)

In [30]:
df.isnull().sum()

,0
customerID,0
Churn,0
customer.gender,0
customer.SeniorCitizen,0
customer.Partner,0
customer.Dependents,0
customer.tenure,0
phone.PhoneService,0
phone.MultipleLines,0
internet.InternetService,0


### Corrección y estandarización de datos

Se estandarizaron las variables categóricas utilizando minúsculas y eliminación
de espacios.  
Se unificaron categorías inconsistentes y se transformó la variable objetivo
`Churn` a formato numérico.  
Además, se crearon variables derivadas para mejorar el análisis.

In [32]:
df["account.Charges.Monthly"].dtype

df["Cuentas_Diarias"] = df["account.Charges.Monthly"] / 30